# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*Answer: Africa

Africa experienced the fastest relative population growth between 1970 and 2022, increasing by approximately 290.4% compared with its 1970 population. Oceania grew by 131.2%, while Europe had the slowest growth at approximately 13.3%. A line chart is appropriate because it shows how population growth changed over time. Indexing each continent to 100 in 1970 makes the growth rates directly comparable.*

In [ ]:
label_countries = [
    'India', 'Bangladesh', 'China', 'Russia',
    'Canada', 'Australia', 'United States'
]

scatter_df = df.copy()

scatter_df['Label'] = scatter_df['Country'].where(
    scatter_df['Country'].isin(label_countries), ''
)

fig = px.scatter(
    scatter_df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    text='Label',
    log_x=True,
    log_y=True,
    title='Land Area versus Population in 2022'
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*The 15 countries with the largest population increases between 1970 and 2022 are:

    India

    China

    Pakistan

    Nigeria

    Indonesia

    United States

    Brazil

    Bangladesh

    Ethiopia

    DR Congo

    Philippines

    Mexico

    Egypt

    Iran

    Vietnam

4 countries are in Africa

The African countries are Nigeria, Ethiopia, DR Congo and Egypt. A sorted horizontal bar chart is suitable because it clearly ranks the countries by their absolute population increase. Different colours distinguish African countries from the others.*

In [ ]:
df['Increase'] = df['2022 Population'] - df['1970 Population']

top15 = df.nlargest(15, 'Increase').sort_values('Increase')

top15['Region group'] = np.where(
    top15['Continent'] == 'Africa',
    'Africa',
    'Other continents'
)

fig = px.bar(
    top15,
    x='Increase',
    y='Country',
    orientation='h',
    color='Region group',
    title='Top 15 Countries by Population Increase (1970–2022)'
)

fig.show()

africa_count = int((top15['Continent'] == 'Africa').sum())
print('African countries in the top 15:', africa_count)



African countries in the top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*There is a positive relationship between land area and population: countries with larger land areas generally tend to have larger populations. However, the relationship is not perfect.

China and India have very large populations, while Bangladesh has a particularly high population relative to its land area. Russia is another interesting case because it has an enormous land area but a comparatively smaller population.

A scatter plot is appropriate because it shows the relationship between two numerical variables. Logarithmic axes make it easier to compare countries with very different sizes and populations.*

In [ ]:
label_countries = [
    'India', 'Bangladesh', 'China', 'Russia',
    'Canada', 'Australia', 'United States'
]

scatter_df = df.copy()

scatter_df['Label'] = scatter_df['Country'].where(
    scatter_df['Country'].isin(label_countries), ''
)

fig = px.scatter(
    scatter_df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    text='Label',
    log_x=True,
    log_y=True,
    title='Land Area versus Population in 2022'
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*Population density varies considerably between continents and between individual countries.

    North America has the highest median density in this dataset, approximately 221.4 people per km².

    Asia follows with approximately 123 people per km².

    Europe has a median of approximately 106.9 people per km².

    South America has the lowest median, approximately 17.8 people per km².

A violin plot is suitable because it displays the distribution and spread of population density rather than only showing the average. A logarithmic scale helps display the large differences between countries.*

In [ ]:
density_order = (
    df.groupby('Continent')['Density']
    .median()
    .sort_values()
    .index.tolist()
)

fig = px.violin(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    category_orders={'Continent': density_order},
    box=True,
    points='outliers',
    title='Population Density Distribution by Continent'
)

fig.update_yaxes(type='log')

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*Between 2000 and 2022, most countries experienced population growth.

    200 countries had population growth.

    34 countries experienced population decline.

    Qatar had the largest percentage increase, approximately 317.2%.

    The Northern Mariana Islands had the largest percentage decline, approximately 38.3%.

A choropleth map is appropriate because it displays the geographical distribution of population growth and decline. A diverging colour scale helps distinguish countries with increasing populations from those with decreasing populations.*

In [ ]:
change_df = df.copy()

change_df['Change (%)'] = (
    change_df['2022 Population'] /
    change_df['2000 Population'] - 1
) * 100

fig = px.choropleth(
    change_df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Change (%)',
    hover_name='Country',
    color_continuous_scale='RdYlBu_r',
    title='Population Change by Country (2000–2022)'
)

fig.show()

print('Largest percentage increases:')
print(
    change_df.nlargest(5, 'Change (%)')[
        ['Country', 'Change (%)']
    ]
)

print(
    'Countries with population decline:',
    int((change_df['Change (%)'] < 0).sum())
)

Largest percentage increases:
                      Country  Change (%)
167                     Qatar  317.242239
219      United Arab Emirates  188.249439
59          Equatorial Guinea  144.520327
215  Turks and Caicos Islands  143.827358
148                     Niger  125.490255
Countries with population decline: 34


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*In 2022, Asia contained the largest share of the world's population, followed by Africa and Europe.

Continent


Share of world population

Asia


59.2%

Africa


17.9%

Europe


9.3%

North America


7.5%

South America


5.5%

Oceania


0.6%

China and India were the two largest individual country populations in 2022.

A treemap is appropriate because it displays the population distribution at two levels: continents and individual countries. The size of each rectangle represents its share of the population.*

In [ ]:
fig = px.treemap(
    df,
    path=[px.Constant('World'), 'Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    title='World Population Composition in 2022'
)

fig.update_traces(
    textinfo='label+percent parent'
)

fig.show()

continent_totals = (
    df.groupby('Continent')['2022 Population']
    .sum()
    .sort_values(ascending=False)
)

print('2022 Population by Continent:')
print(continent_totals)

2022 Population by Continent:
Continent
Asia             4721383274
Africa           1426730932
Europe            743147538
North America     600296136
South America     436816608
Oceania            45038554
Name: 2022 Population, dtype: int64


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*The world population shares changed between 1970 and 2022 as follows:

Region


1970


2022

Africa


9.89%


17.89%

Asia


58.06%


59.21%

Europe


17.76%


9.32%

Rest of the world


14.29%


13.57%

Africa experienced the largest increase in its share of world population, rising by approximately 8 percentage points. Asia remained relatively stable, while Europe's share decreased significantly.

A 100% stacked bar chart is appropriate because it compares the proportion of world population represented by each region in 1970 and 2022. Grouping the other three continents into the Rest of the world category makes the comparison clearer.*

In [ ]:
years = [1970, 2022]

cols = [f'{year} Population' for year in years]

totals_by_continent = df.groupby('Continent')[cols].sum()

world_totals = df[cols].sum()

shares = totals_by_continent.div(
    world_totals, axis=1
) * 100

share_rows = []

for year in years:
    for continent in [
        'Africa', 'Asia', 'Europe',
        'North America', 'Oceania', 'South America'
    ]:
        group = (
            continent
            if continent in ['Africa', 'Asia', 'Europe']
            else 'Rest of the world'
        )

        share_rows.append({
            'Year': str(year),
            'Group': group,
            'Share (%)': shares.loc[
                continent, f'{year} Population'
            ]
        })

share_df = pd.DataFrame(share_rows).groupby(
    ['Year', 'Group'],
    as_index=False
)['Share (%)'].sum()

fig = px.bar(
    share_df,
    x='Year',
    y='Share (%)',
    color='Group',
    barmode='stack',
    title='World Population Share by Continent: 1970 vs 2022'
)

fig.show()

print(share_df.pivot(
    index='Group',
    columns='Year',
    values='Share (%)'
).round(2))

Year                1970   2022
Group                          
Africa              9.89  17.89
Asia               58.06  59.21
Europe             17.76   9.32
Rest of the world  14.29  13.57
